# Build Your Own Image Classifier

Last week you used **Teachable Machine**: point a webcam, click "train", get a
classifier. It worked in seconds, with almost no data. Today you're going to
build the same trick yourself, in code, and see exactly why it works.

The short answer: Teachable Machine doesn't actually learn to see from
scratch. It borrows a network that already learned to see from **1.4 million**
photos (ImageNet), and only trains a small layer on top for *your* classes.
That's called **transfer learning**, and it's the whole reason a few dozen
photos and a couple of minutes are enough.

This time, nothing is hidden:
- **You** pick the classes.
- **You** collect the images (a live search, not a curated dataset someone
  handed you).
- **You** can read every line that trains the model and makes a prediction.

That distinction matters. A model you download from somewhere is a black
box unless you go looking — you don't know what data trained it or what's
actually inside the file. A model you build yourself, from images you can
see, isn't. Step 9 pushes on exactly this, using the same pipeline for
something less comfortable than dog breeds.

## Step 0 — Setup

Run this once. `tensorflow` and `matplotlib` already exist in Colab — only
`ddgs` (image search) and `gradio` (the live demo at the end) need installing.

In [ ]:
!pip install -q ddgs gradio

import io
import shutil
from pathlib import Path

import numpy as np
import requests
import tensorflow as tf
import matplotlib.pyplot as plt
from PIL import Image
from ddgs import DDGS

print("TensorFlow:", tf.__version__)

## Step 1 — Pick your classes

Pick 2–4 things a camera can tell apart. Good choices are visually distinct
but not *trivially* distinct — two dog breeds, two chair styles, two of your
own hand gestures. Avoid single ambiguous words (`"mouse"` will get you both
animals and computer mice).

The dictionary key becomes the folder/class name (no spaces). The value is
the actual search phrase — adding a word like `"photo"` usually filters out
cartoons/logos and gets you real pictures. There's no starter example this
time — write your own.

In [ ]:
# TODO: add 2-4 entries. Key = folder name (no spaces), value = search phrase.
# Shape (write your own values, don't copy this): "cat": "cat photo"
CLASSES = {

}

assert CLASSES, "Add at least 2 classes to CLASSES above before continuing"

IMAGES_PER_CLASS = 100
DATA_DIR = Path("data")

## Step 2 — Scrape the images

This searches DuckDuckGo images for each class and downloads the first
`IMAGES_PER_CLASS` results, re-saving every one as a clean RGB JPEG (some
search results are broken links, weird formats, or corrupt — those are
silently skipped, which is why you may end up with fewer than you asked
for).

**Expect some junk in the results.** That's not a bug — you'll deal with it
in Step 3. It's also a small, low-stakes preview of Step 9's theme: you
can't fully trust what you didn't personally check.

`scrape_class` below is provided — it's plumbing (HTTP requests, error
handling), not the interesting part. The loop that actually uses it, right
after, is on you.

In [ ]:
HEADERS = {"User-Agent": "Mozilla/5.0"}

def scrape_class(query, out_dir, n):
    out_dir.mkdir(parents=True, exist_ok=True)
    with DDGS() as ddgs:
        results = list(ddgs.images(query, max_results=n))
    saved = 0
    for r in results:
        url = r.get("image")
        if not url:
            continue
        try:
            resp = requests.get(url, headers=HEADERS, timeout=6)
            img = Image.open(io.BytesIO(resp.content)).convert("RGB")
            img.save(out_dir / f"{saved:03d}.jpg", "JPEG", quality=90)
            saved += 1
        except Exception:
            continue
    return saved

if DATA_DIR.exists():
    shutil.rmtree(DATA_DIR)

# TODO: for each (folder_name, query) pair in CLASSES, call
# scrape_class(query, DATA_DIR / folder_name, IMAGES_PER_CLASS), store the
# result in n, and print how many images that class actually got.
for folder_name, query in CLASSES.items():
    n = ___
    print(f"{folder_name}: {n} images saved")

## Step 3 — Look at what you actually got

Before training on it, look at it — same instinct as checking a downloaded
model before you trust it. This grid shows a random sample from each class.

In [ ]:
def show_samples(data_dir, n=6):
    class_dirs = sorted(d for d in Path(data_dir).iterdir() if d.is_dir())
    fig, axes = plt.subplots(len(class_dirs), n, figsize=(n * 2, len(class_dirs) * 2))
    for row, cdir in enumerate(class_dirs):
        files = list(cdir.glob("*.jpg"))
        sample = np.random.choice(files, size=min(n, len(files)), replace=False)
        for col in range(n):
            ax = axes[row][col] if len(class_dirs) > 1 else axes[col]
            ax.axis("off")
            if col < len(sample):
                ax.imshow(Image.open(sample[col]))
                if col == 0:
                    ax.set_title(cdir.name, loc="left", fontsize=10)
    plt.tight_layout()
    plt.show()

show_samples(DATA_DIR)

See junk (wrong subject, a logo, a screenshot, a diagram instead of a
photo)? Delete it. Easiest way: open the **Files** pane on the left, find the
file under `data/<class_name>/`, right-click → delete. Or list the bad files
below and run the cell — either way, re-run the sample grid above afterward
to confirm.

In [ ]:
# Example: BAD_FILES = ["golden_retriever/013.jpg", "poodle/047.jpg"]
BAD_FILES = []

for f in BAD_FILES:
    (DATA_DIR / f).unlink(missing_ok=True)

print(f"Deleted {len(BAD_FILES)} file(s)")

## Step 4 — Turn the folders into a dataset

Standard Keras helper: point it at a folder of `class_name/image.jpg`
folders, it figures out the classes and splits off 20% for validation
(images the model never trains on, used to check it actually generalizes).

In [ ]:
IMG_SIZE = (160, 160)
BATCH_SIZE = 16

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset="training", seed=42,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE,
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset="validation", seed=42,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE,
)

class_names = train_ds.class_names
print("Classes:", class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(200).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)

## Step 5 — Borrow a pair of eyes

This is the actual trick behind Teachable Machine. `MobileNetV2` below
already knows how to see — it was trained on 1.4 million ImageNet photos to
tell apart 1000 categories. We freeze it (`trainable = False`, so none of
that knowledge gets overwritten) and only train a small new head on top for
*your* classes.

`data_augmentation` randomly flips/rotates/zooms training images each epoch
— it's a cheap way to squeeze more variety out of ~100 photos, and the same
idea (augmentation matters for small datasets) shows up in the ESP32 digit
project if you look at that repo later.

Two things below are on you:
- Should `base_model.trainable` be `True` or `False`? (Do we want the
  1.4-million-photo knowledge to keep changing while we train, or stay put?)
- The last layer needs to output one score per class, turned into
  probabilities that add up to 1 — that activation is called `softmax`. How
  many outputs does that layer need?

In [ ]:
num_classes = len(class_names)

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
])

base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,), include_top=False, weights="imagenet"
)
base_model.trainable = ___  # True or False?

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(___, activation="softmax")(x)  # how many outputs?
model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()

## Step 6 — Train

On ~100 images per class this should take well under a minute per epoch on
Colab's CPU. Pick a number of epochs (full passes over the training data)
below — try something between 5 and 15. More epochs means more training
time, not automatically more accuracy; the plot after training shows you
whether it was worth it.

In [ ]:
EPOCHS = ___  # pick a number, e.g. between 5 and 15

history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS)

plt.plot(history.history["accuracy"], label="train accuracy")
plt.plot(history.history["val_accuracy"], label="val accuracy")
plt.xlabel("epoch")
plt.legend()
plt.title("How fast did it learn?")
plt.show()

## Step 7 — Where does it get it wrong?

The interesting cases are the mistakes, not the correct predictions. Often
they trace straight back to a bad photo you missed in Step 3 — mislabeled,
wrong subject, or genuinely ambiguous.

In [ ]:
wrong = []
for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    pred_labels = np.argmax(preds, axis=1)
    for img, true, pred in zip(images, labels.numpy(), pred_labels):
        if true != pred:
            wrong.append((img.numpy().astype("uint8"), class_names[true], class_names[pred]))

print(f"{len(wrong)} mistake(s) out of the validation set")

n_show = min(6, len(wrong))
if n_show:
    fig, axes = plt.subplots(1, n_show, figsize=(n_show * 2, 2))
    axes = [axes] if n_show == 1 else axes
    for i in range(n_show):
        img, true, pred = wrong[i]
        axes[i].imshow(img)
        axes[i].axis("off")
        axes[i].set_title(f"real: {true}\nguess: {pred}", fontsize=8)
    plt.tight_layout()
    plt.show()

## Step 8 — Try it live

Point your webcam at something, or upload a photo, and watch your own model
guess.

In [ ]:
import gradio as gr

def predict(img):
    if img is None:
        return {}
    img = Image.fromarray(img).convert("RGB").resize(IMG_SIZE)
    arr = np.expand_dims(np.array(img), axis=0).astype("float32")
    preds = model.predict(arr, verbose=0)[0]
    return {class_names[i]: float(preds[i]) for i in range(len(class_names))}

demo = gr.Interface(
    fn=predict,
    inputs=gr.Image(sources=["webcam", "upload"], type="numpy"),
    outputs=gr.Label(num_top_classes=num_classes),
    title="Your Classifier",
)
demo.launch(debug=True)

## Step 9 — Assignment: Dark Technology

The pipeline you just used — search for images, train a classifier, ship
it — is also exactly how real, harmful classifiers get built. A well-known
case: in 2015, Google Photos' auto-tagging system labelled photos of Black
people as gorillas. Nobody set out to build that — it came from what wasn't
in the training data. The tool wasn't the problem; the data, and what got
done with the output, was.

**Your assignment:** come up with your own idea for something a classifier
could do that counts as *Dark Technology* — a real capability you could
actually build with what you now know, that should make people
uncomfortable if it got shipped for real. It doesn't have to copy the Google
example. Think about surveillance, profiling, exclusion, inferring
something private from an image, or a classifier that only works well for
one group of people. The point is: what makes your idea "dark," and why is
it this easy to build?

Then **build it**. Copy this notebook, swap in your own classes and search
terms, and get it actually running end to end — via the live demo in Step 8.
It doesn't need to be polished. It needs to work.

You'll show it to the class afterward — 2-3 minutes each:
- What did you build?
- Why does it count as Dark Technology?
- What would have to be different about the data, or the process, to stop
  this from happening for real?

Stuck for an idea? Some starting angles:
- A classifier that infers something about a person they didn't choose to
  reveal.
- A classifier trained on a narrow slice of people, then used on everyone.
- A classifier whose mistakes hurt one group far more than another.

## Stretch goals (if you finish early)

- Add a 3rd or 4th class.
- Try `MobileNetV3Small` or `EfficientNetB0` instead of `MobileNetV2` — does
  accuracy or speed change?
- Unfreeze the last few layers of `base_model` and fine-tune at a very low
  learning rate (`1e-5`) for a couple more epochs — usually a small accuracy
  bump, at the cost of much slower training.
- `model.save("my_model.keras")` and download it — the ESP32 digit-recognition
  repo (if your instructor shares it) shows what the *next* step looks like:
  shrinking a Keras model down to run entirely on a $5 microcontroller,
  no cloud at all.